# SemanticChunker (의미 기반 분할)

텍스트를 **의미적 유사성**에 기반하여 분할합니다.

**Reference**
- [Greg Kamradt의 노트북](https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/tutorials/LevelsOfTextSplitting/5_Levels_Of_Text_Splitting.ipynb)

텍스트를 문장 단위로 나눈 뒤, 앞뒤 문장을 묶어 임베딩하고, **인접한 문장 사이의 임베딩 거리가 크게 벌어지는 지점**에서 청크를 나눕니다.

> **🔄 최신 상황 (중요)**
>
> 책에서 사용한 `langchain_experimental.text_splitter.SemanticChunker` 가 들어 있던 **`langchain-experimental` 패키지는 2026년 5월 공식적으로 지원 종료(sunset)** 되었고,
> GitHub 저장소도 보관(archive) 처리되었습니다. `SemanticChunker` 는 `langchain-text-splitters` 로 이전되지 않았으며, 공식 문서의 텍스트 분할기 목록에서도 빠졌습니다.
>
> LangChain 팀은 지원 종료 공지에서 이런 실험적 기능은 **애플리케이션 코드에 직접 구현하는 방향**을 안내하고 있습니다.
> 그래서 이 노트북은 다음과 같이 구성했습니다.
>
> 1. **(권장)** `langchain-text-splitters` 의 `TextSplitter` 를 상속해 **같은 알고리즘을 직접 구현** → `split_text`, `create_documents`, `split_documents` 를 그대로 사용 가능
> 2. **(레거시)** 기존 코드를 당장 돌려야 할 때 `langchain-experimental==0.4.2` 를 고정 설치하는 방법 (맨 아래)
>
> 그 밖의 변경점
> - `OpenAIEmbeddings()` 의 모델을 **명시**합니다(`text-embedding-3-small`). 기본값에 의존하지 않는 것이 권장 방식입니다.
> - 파일 읽기 시 `encoding="utf-8"` 명시
> - 세 가지 임계값 방식을 비교할 때 **임베딩을 한 번만 계산해 재사용**하도록 하여 API 비용을 줄였습니다.

In [ ]:
%pip install -qU langchain-text-splitters langchain-openai python-dotenv numpy

샘플 텍스트를 로드합니다.

In [ ]:
from pathlib import Path

file = Path("./data/appendix-keywords.txt").read_text(encoding="utf-8")
print(file[:350])

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정 (.env 파일의 OPENAI_API_KEY 로드)
from dotenv import load_dotenv

load_dotenv()

## 임베딩 모델 준비

🔄 모델명을 명시합니다. 다른 제공자의 임베딩(예: `langchain-google-genai`, `langchain-huggingface` 등 파트너 패키지)으로 바꿔도
`Embeddings` 인터페이스(`embed_documents`)만 같으면 아래 분할기가 그대로 동작합니다.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## SemanticChunker 직접 구현

`langchain_experimental` 의 알고리즘을 그대로 따르되, 핵심만 남긴 구현입니다.

1. **문장 분리**: 정규식 `(?<=[.?!])\s+` 로 문장을 나눕니다.
2. **문맥 결합**: 각 문장에 앞뒤 `buffer_size` 개 문장을 붙여 임베딩합니다. (문장 하나만 임베딩하면 노이즈가 크기 때문)
3. **거리 계산**: 인접한 결합 문장 임베딩 간 **코사인 거리**(1 - 코사인 유사도)를 구합니다.
4. **분할 지점 결정**: 거리가 임계값을 넘는 곳에서 자릅니다. 임계값 방식은 3가지입니다.
   - `percentile`: 거리 분포의 N 백분위수 (기본 95)
   - `standard_deviation`: 평균 + N × 표준편차 (기본 3)
   - `interquartile`: 평균 + N × 사분위범위(IQR) (기본 1.5)
5. 🔄 **(추가 기능) `min_chunk_size`**: 너무 짧은 청크를 다음 청크와 합칩니다. 의미 분할은 조각이 지나치게 작아지는 경향이 있어 실무에서 자주 필요합니다.

`TextSplitter` 를 상속했기 때문에 `create_documents()`, `split_documents()` 등 다른 분할기와 **같은 인터페이스**로 사용할 수 있습니다.

In [ ]:
import re
from typing import Literal

import numpy as np
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import TextSplitter

BreakpointType = Literal["percentile", "standard_deviation", "interquartile"]

DEFAULT_THRESHOLD_AMOUNTS: dict[str, float] = {
    "percentile": 95,
    "standard_deviation": 3,
    "interquartile": 1.5,
}


class SemanticChunker(TextSplitter):
    """임베딩 거리 기반 의미 분할기 (langchain_experimental.SemanticChunker 대체 구현)."""

    def __init__(
        self,
        embeddings: Embeddings,
        buffer_size: int = 1,
        breakpoint_threshold_type: BreakpointType = "percentile",
        breakpoint_threshold_amount: float | None = None,
        sentence_split_regex: str = r"(?<=[.?!])\s+",
        min_chunk_size: int | None = None,
        cache: dict | None = None,
        **kwargs,
    ) -> None:
        super().__init__(**kwargs)
        if breakpoint_threshold_type not in DEFAULT_THRESHOLD_AMOUNTS:
            raise ValueError(f"지원하지 않는 방식: {breakpoint_threshold_type}")
        self.embeddings = embeddings
        self.buffer_size = buffer_size
        self.breakpoint_threshold_type = breakpoint_threshold_type
        self.breakpoint_threshold_amount = (
            breakpoint_threshold_amount
            if breakpoint_threshold_amount is not None
            else DEFAULT_THRESHOLD_AMOUNTS[breakpoint_threshold_type]
        )
        self.sentence_split_regex = sentence_split_regex
        self.min_chunk_size = min_chunk_size
        # 같은 텍스트에 대한 임베딩 거리를 재사용하기 위한 캐시 (여러 분할기가 공유 가능)
        self._cache = cache if cache is not None else {}

    # 1) 문장 분리
    def _split_sentences(self, text: str) -> list[str]:
        return [s for s in re.split(self.sentence_split_regex, text) if s.strip()]

    # 2) + 3) 문맥 결합 후 인접 문장 간 코사인 거리 계산
    def calculate_distances(self, text: str) -> tuple[list[str], np.ndarray]:
        if text in self._cache:
            return self._cache[text]

        sentences = self._split_sentences(text)
        b = self.buffer_size
        combined = [
            " ".join(sentences[max(0, i - b) : i + b + 1]) for i in range(len(sentences))
        ]
        vectors = np.array(self.embeddings.embed_documents(combined))
        vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
        similarities = np.sum(vectors[:-1] * vectors[1:], axis=1)
        distances = 1.0 - similarities

        self._cache[text] = (sentences, distances)
        return sentences, distances

    # 4) 임계값 계산
    def _threshold(self, distances: np.ndarray) -> float:
        amount = self.breakpoint_threshold_amount
        if self.breakpoint_threshold_type == "percentile":
            return float(np.percentile(distances, amount))
        if self.breakpoint_threshold_type == "standard_deviation":
            return float(np.mean(distances) + amount * np.std(distances))
        q1, q3 = np.percentile(distances, [25, 75])
        return float(np.mean(distances) + amount * (q3 - q1))

    # 5) 너무 작은 청크 병합
    def _merge_small(self, chunks: list[str]) -> list[str]:
        if not self.min_chunk_size:
            return chunks
        merged: list[str] = []
        buffer = ""
        for chunk in chunks:
            buffer = f"{buffer} {chunk}".strip()
            if self._length_function(buffer) >= self.min_chunk_size:
                merged.append(buffer)
                buffer = ""
        if buffer:
            if merged:
                merged[-1] = f"{merged[-1]} {buffer}"
            else:
                merged.append(buffer)
        return merged

    def split_text(self, text: str) -> list[str]:
        sentences = self._split_sentences(text)
        if len(sentences) <= 1:
            return sentences

        sentences, distances = self.calculate_distances(text)
        threshold = self._threshold(distances)
        breakpoints = np.where(distances > threshold)[0]

        chunks: list[str] = []
        start = 0
        for idx in breakpoints:
            chunks.append(" ".join(sentences[start : idx + 1]))
            start = idx + 1
        if start < len(sentences):
            chunks.append(" ".join(sentences[start:]))
        return self._merge_small(chunks)

## 텍스트 분할

기본 설정(`percentile`, 95)으로 분할기를 만들고 `split_text()` 로 분할합니다.

In [ ]:
# 세 가지 임계값 방식을 비교할 때 임베딩을 다시 계산하지 않도록 캐시를 공유합니다.
shared_cache: dict = {}

text_splitter = SemanticChunker(embeddings, cache=shared_cache)
chunks = text_splitter.split_text(file)
print(len(chunks))

In [ ]:
print(chunks[0])

`create_documents()` 로 `Document` 리스트를 만들 수 있습니다. (`TextSplitter` 상속 덕분에 별도 구현 없이 사용 가능)

In [ ]:
docs = text_splitter.create_documents([file])
print(docs[0].page_content)

## Breakpoints (분할 지점)

인접 문장 간 임베딩 거리가 임계값을 넘으면 그 지점에서 분리합니다.

- 참고 영상: https://youtu.be/8OJC21T2SL4?si=PzUtNGYJ_KULq3-w&t=2580

🔄 거리 분포를 직접 확인해 보면 임계값 설정이 어떻게 작용하는지 이해하기 쉽습니다.

In [ ]:
sentences, distances = text_splitter.calculate_distances(file)
print(f"문장 수: {len(sentences)}, 거리 개수: {len(distances)}")
print(f"거리 최소/평균/최대: {distances.min():.3f} / {distances.mean():.3f} / {distances.max():.3f}")
for p in (50, 70, 95):
    print(f"{p} 백분위수: {np.percentile(distances, p):.3f}")

In [ ]:
def show_chunks(docs, n: int = 5) -> None:
    for i, doc in enumerate(docs[:n]):
        print(f"[Chunk {i}]", end="\n\n")
        print(doc.page_content)
        print("===" * 20)

### Percentile

모든 인접 거리 중 지정한 백분위수보다 큰 지점에서 분리합니다. 값이 **낮을수록 더 많이** 쪼개집니다.

In [ ]:
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70,
    cache=shared_cache,  # 🔄 임베딩 재사용 (추가 API 호출 없음)
)
docs = text_splitter.create_documents([file])
show_chunks(docs)

In [ ]:
print(len(docs))

### Standard Deviation

`평균 + breakpoint_threshold_amount × 표준편차` 보다 큰 거리에서 분리합니다.

In [ ]:
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="standard_deviation",
    breakpoint_threshold_amount=1.25,
    cache=shared_cache,
)
docs = text_splitter.create_documents([file])
show_chunks(docs)

In [ ]:
print(len(docs))

### Interquartile

`평균 + breakpoint_threshold_amount × IQR(Q3 - Q1)` 보다 큰 거리에서 분리합니다.

In [ ]:
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="interquartile",
    breakpoint_threshold_amount=0.5,
    cache=shared_cache,
)
docs = text_splitter.create_documents([file])
show_chunks(docs)

In [ ]:
print(len(docs))

### 🔄 추가: 최소 청크 크기로 조각 병합

의미 기반 분할은 짧은 조각을 많이 만드는 경향이 있어, 너무 짧은 청크는 검색 품질을 떨어뜨릴 수 있습니다.
`min_chunk_size` 를 지정하면 기준 길이(기본은 문자 수, `length_function` 으로 변경 가능)에 도달할 때까지 인접 청크를 합칩니다.

In [ ]:
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70,
    min_chunk_size=300,
    cache=shared_cache,
)
docs = text_splitter.create_documents([file])
print(len(docs))
print("가장 짧은 청크 길이:", min(len(d.page_content) for d in docs))

## (레거시) langchain-experimental 을 그대로 쓰는 방법

책의 코드를 수정 없이 실행해야 한다면 마지막 배포 버전을 **고정 설치**할 수 있습니다. 인터페이스는 위 구현과 같습니다.
다만 더 이상 유지보수·보안 패치가 이루어지지 않으므로 **새 프로젝트에는 권장하지 않습니다.**

```python
%pip install "langchain-experimental==0.4.2"

from langchain_experimental.text_splitter import SemanticChunker
text_splitter = SemanticChunker(
    OpenAIEmbeddings(model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70,
)
```